Check how many questions and KCs have 90% of correct response
- Look at guidelines for learning engineering
- How CMU suggests learning with PyBKT, how they suggest guidelines for creating courses. Look at pyBKT extended paper.
- Do the same analysis on both aggregation types both combined and aggregated.


This is counterintuitive but important. If you had to choose between giving each student 10 more questions per KC or enrolling 10 more students, enrolling more students does more to improve BKT reliability. More students is more valuable than more questions per student.

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

base_path = Path().resolve().parent
file_path = base_path / 'data' / 'raw' 

In [4]:
data = pd.read_excel(file_path / 'Stellar_edu_MDS_ap_stats_dataset - v1.9.xlsx', sheet_name='Student_Observations')

In [ ]:
# new KC Maps
mkc_maps = pd.read_excel(file_path / 'mkc_mapping_pack_v1.0..xlsx', sheet_name='FineKC_to_ModelingKC_Map')

In [7]:
mkc_map = mkc_maps[['fine_kc_id', 'modeling_kc_id']]

In [8]:
mkc_map.nunique()

fine_kc_id        256
modeling_kc_id     47
dtype: int64

In [9]:
data_mkc = data.copy()

In [3]:
data['primary_kc_id'].nunique()

236

In [4]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 21808 entries, 0 to 21807
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   student_id                21808 non-null  str    
 1   assignment_id             21808 non-null  str    
 2   class_num                 21808 non-null  int64  
 3   observation_id            21808 non-null  str    
 4   item_type                 21808 non-null  str    
 5   source_question           21808 non-null  str    
 6   primary_kc_id             21808 non-null  str    
 7   all_kc_ids                21808 non-null  str    
 8   score                     21808 non-null  float64
 9   max_score                 21808 non-null  int64  
 10  percent_score             21808 non-null  int64  
 11  student_response          21808 non-null  str    
 12  correct_answer_or_rubric  21808 non-null  str    
 13  rubric_level              4642 non-null   str    
dtypes: float64(1), in

In [5]:
data.head()

,student_id,assignment_id,class_num,observation_id,item_type,source_question,primary_kc_id,all_kc_ids,score,max_score,percent_score,student_response,correct_answer_or_rubric,rubric_level
0,S001,HW1,1,HW1_PCA_Q01,MCQ,PCA Q01,KC.U1.02.observational_unit_variable,KC.U1.02.observational_unit_variable,0.0,1,0,C,E,NaN
1,S001,HW1,1,HW1_PCA_Q02,MCQ,PCA Q02,KC.U1.03.variable_type_cat_quant,KC.U1.03.variable_type_cat_quant,1.0,1,100,E,E,NaN
2,S001,HW1,1,HW1_PCA_Q03,MCQ,PCA Q03,KC.U1.03.variable_type_cat_quant,KC.U1.03.variable_type_cat_quant,0.0,1,0,C,D,NaN
3,S001,HW1,1,HW1_PCA_Q04,MCQ,PCA Q04,KC.U1.05.categorical_freq_relative,KC.U1.05.categorical_freq_relative,1.0,1,100,E,E,NaN
4,S001,HW1,1,HW1_PCA_Q05,MCQ,PCA Q05,KC.U1.05.categorical_freq_relative,KC.U1.05.categorical_freq_relative,1.0,1,100,B,B,NaN


In [18]:
data.columns

Index(['student_id', 'assignment_id', 'class_num', 'observation_id',
       'item_type', 'source_question', 'primary_kc_id', 'all_kc_ids', 'score',
       'max_score', 'percent_score', 'student_response',
       'correct_answer_or_rubric', 'rubric_level'],
      dtype='str')

In [6]:
data[data['score'] == 0.5]

,student_id,assignment_id,class_num,observation_id,item_type,source_question,primary_kc_id,all_kc_ids,score,max_score,percent_score,student_response,correct_answer_or_rubric,rubric_level
66,S001,HW3,3,HW3_PCFRQ_Q1c,FRQ_Component,PCFRQ Q1c,KC.U1.20.normal_model_appropriateness,KC.U1.20.normal_model_appropriateness|KC.U1.10...,0.5,1,50,P,Rubric: no; not mound-shaped/normal; justify u...,P
113,S002,HW2,2,HW2_SGFRQ1_Q1b,FRQ_Component,SGFRQ1 Q1b,KC.U1.16.resistance_skew_mean_median,KC.U1.16.resistance_skew_mean_median|KC.U1.10....,0.5,1,50,P,Rubric: mean overestimates typical price becau...,P
136,S002,HW3,3,HW3_PCFRQ_Q2a,FRQ_Component,PCFRQ Q2a,KC.U1.19.z_score_compute_interpret,KC.U1.19.z_score_compute_interpret,0.5,1,50,P,Rubric: z = 1 for north and z = -1.2 for south...,P
205,S003,HW3,3,HW3_PCFRQ_Q2a,FRQ_Component,PCFRQ Q2a,KC.U1.19.z_score_compute_interpret,KC.U1.19.z_score_compute_interpret,0.5,1,50,P,Rubric: z = 1 for north and z = -1.2 for south...,P
251,S004,HW2,2,HW2_SGFRQ1_Q1b,FRQ_Component,SGFRQ1 Q1b,KC.U1.16.resistance_skew_mean_median,KC.U1.16.resistance_skew_mean_median|KC.U1.10....,0.5,1,50,P,Rubric: mean overestimates typical price becau...,P
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21747,S025,MF1,27,MF1_FRQ6E,FRQ_Component,Investigative Q1,KC.U10.07.bootstrap_interval_interpret_context,KC.U10.07.bootstrap_interval_interpret_context...,0.5,1,50,P,Rubric: use percentile endpoints and interpret...,P
21791,S025,MF2,27,MF2_FRQ2A,FRQ_Component,FRQ 2,KC.U7.20.two_sample_t_interval_compute_interpret,KC.U7.20.two_sample_t_interval_compute_interpr...,0.5,1,50,P,Rubric: select two-sample or randomized-compar...,P
21794,S025,MF2,27,MF2_FRQ3A,FRQ_Component,FRQ 1,KC.U4.16.expected_value_calculation,KC.U4.16.expected_value_calculation|KC.U4.17.e...,0.5,1,50,P,Rubric: use n times p and state the expected n...,P
21801,S025,MF2,27,MF2_FRQ5B,FRQ_Component,FRQ 1,KC.U9.19.lsrl_equation_from_data,KC.U9.19.lsrl_equation_from_data|KC.U9.14.slop...,0.5,1,50,P,Rubric: write LSRL equation and state H0/H_a f...,P


In [12]:
question_difficulty = (data[data['score'] != 0.5]
    .assign(correct=lambda df: df['score'].astype(int))
    .groupby('observation_id')       
    .agg(
        correct_rate = ('correct', 'mean'),
        n_students   = ('correct', 'count')
    )
    .reset_index()
    .sort_values('correct_rate', ascending=False)
)

question_difficulty.head()
len(question_difficulty)

895

In [11]:
too_easy_questions = question_difficulty[question_difficulty['correct_rate'] >= 0.80]
print(f"Questions with ≥ 80% correct rate: {len(too_easy_questions)}")
print(too_easy_questions)

Questions with ≥ 80% correct rate: 24
        observation_id  correct_rate  n_students
349        HW2_PCA_Q13      0.880000          25
511       HW7_U4PA_Q05      0.880000          25
561       HW9_U4FRQ2_a      0.857143          14
47    HW11_U5FRQ2_Q2ci      0.842105          19
572       HW9_U4PB_Q10      0.840000          25
500      HW7_SG34_Q112      0.840000          25
182        HW1_PCA_Q02      0.840000          25
193        HW1_SG1_Q24      0.840000          25
72        HW13_U5C_Q06      0.833333          24
576          ME1_FRQ1a      0.823529          17
61    HW12_U5FRQ1_Q1a1      0.823529          17
149  HW17_U6FRQ2_Q2d_i      0.812500          16
627         ME2_FRQ04a      0.812500          16
787          MF1_FRQ5A      0.800000          15
502      HW7_SG34_Q120      0.800000          25
355        HW2_PCB_Q01      0.800000          25
381      HW3_PCFRQ_Q1c      0.800000          20
401      HW4_U2PCA_Q05      0.800000          25
16       HW10_U4PC_Q01      0.8

In [44]:
hard_questions = question_difficulty[question_difficulty['correct_rate']<= 0.30]
print(f"Questions with ≤ 30% correct rate: {len(hard_questions)}")
print(hard_questions)

Questions with ≤ 30% correct rate: 38
          observation_id  correct_rate  n_students
831            MF1_MCQ37      0.291667          24
818            MF1_MCQ24      0.291667          24
798            MF1_MCQ04      0.291667          24
796            MF1_MCQ02      0.291667          24
824            MF1_MCQ30      0.291667          24
889            MF2_MCQ35      0.291667          24
435      HW4_U2PCFRQ_Q2d      0.291667          24
849            MF2_FRQ5C      0.285714          21
740            ME4_MCQ21      0.280000          25
733            ME4_MCQ14      0.280000          25
613            ME1_MCQ26      0.280000          25
837            MF2_FRQ1C      0.277778          18
329       HW27_PWR_FRQ18      0.277778          18
778            MF1_FRQ2A      0.266667          15
839            MF2_FRQ2B      0.266667          15
221        HW20_U7FRQ1_b      0.263158          19
250         HW22_U8B_Q11      0.260870          23
248         HW22_U8B_Q09      0.260870      

In [13]:
question_difficulty.to_csv(base_path / 'data' / 'processed' / 'question_difficulty.csv', index=False)

In [35]:
kc_difficulty = (data[data['score'] != 0.5]
    .assign(correct=lambda df: df['score'].astype(int))
    .groupby('primary_kc_id')       
    .agg(
        correct_rate = ('correct', 'mean'),
        n_obs   = ('correct', 'count'),
        n_students   = ('student_id', 'nunique')
    )
    .reset_index()
    .sort_values('correct_rate', ascending=False)
)

kc_difficulty.head()
len(kc_difficulty)

236

In [41]:
kc_difficulty.head(236)

,primary_kc_id,correct_rate,n_obs,n_students
124,KC.U5.06.point_estimator_statistic_parameter,0.746269,67,24
6,KC.U1.08.graph_choice_categorical,0.740000,50,25
28,KC.U10.03.bootstrap_same_sample_size,0.739130,23,23
50,KC.U2.08.explanatory_response_variables,0.720000,25,25
205,KC.U8.10.chi_square_distribution_shape,0.708333,24,24
...,...,...,...,...
191,KC.U7.21.two_sample_t_interval_claim_plausibility,0.347826,46,23
59,KC.U2.18.standard_deviation_residuals,0.323077,65,25
197,KC.U7.27.multiple_comparisons_alpha_adjustment,0.294118,34,24
40,KC.U10.15.rejection_region_power_calculation,0.290909,55,24


In [42]:
kc_difficulty.to_csv(base_path / 'data' / 'processed' / 'kc_difficulty.csv', index=False)

In [37]:
too_easy_kc = kc_difficulty[kc_difficulty['correct_rate'] >= 0.70]
print(f"Knowledge components with ≥ 70% correct rate: {len(too_easy_kc)}")
too_easy_kc

Knowledge components with ≥ 70% correct rate: 6


,primary_kc_id,correct_rate,n_obs,n_students
124,KC.U5.06.point_estimator_statistic_parameter,0.746269,67,24
6,KC.U1.08.graph_choice_categorical,0.740000,50,25
28,KC.U10.03.bootstrap_same_sample_size,0.739130,23,23
50,KC.U2.08.explanatory_response_variables,0.720000,25,25
205,KC.U8.10.chi_square_distribution_shape,0.708333,24,24
180,KC.U7.09.generalization_scope_quantitative_inf...,0.700000,10,10


In [40]:
too_hard_kc = kc_difficulty[kc_difficulty['correct_rate'] <= 0.30]
print(f"Knowledge components with ≤ 30% correct rate: {len(too_hard_kc)}")
too_hard_kc

Knowledge components with ≤ 30% correct rate: 3


,primary_kc_id,correct_rate,n_obs,n_students
197,KC.U7.27.multiple_comparisons_alpha_adjustment,0.294118,34,24
40,KC.U10.15.rejection_region_power_calculation,0.290909,55,24
176,KC.U7.05.one_sample_t_interval_compute,0.270270,37,23


In [21]:
total_students = data['student_id'].nunique()
total_questions = data['source_question'].nunique()
total_kcs = data['primary_kc_id'].nunique()
total_obs = len(data)

print(f"Total Unique Students: {total_students}")
print(f"Total Unique Questions (Items): {total_questions}")
print(f"Total Unique KCs: {total_kcs}")
print(f"Total Graded Observations: {total_obs}\n")

Total Unique Students: 25
Total Unique Questions (Items): 640
Total Unique KCs: 236
Total Graded Observations: 21808



In [33]:
kc_success = data.groupby('primary_kc_id')['score'].mean()

print("Validating Data to Original pyBKT Recommendations:")
# 1. Vertical Depth: How many unique students attempted each KC?
students_per_kc = data.groupby('primary_kc_id')['student_id'].nunique()
low_depth_kcs = students_per_kc[students_per_kc < 100] 

# 2. Horizontal Sequence: Average number of practice opportunities (attempts) a student gets per KC
attempts_per_student_per_kc = data.groupby(['primary_kc_id', 'student_id']).size().reset_index(name='sequence_length')

avg_sequence_per_kc = attempts_per_student_per_kc.groupby('primary_kc_id')['sequence_length'].mean()
short_sequence_kcs = avg_sequence_per_kc[avg_sequence_per_kc < 15]  

print(f"KCs with low vertical depth (< 100 students): {len(low_depth_kcs)} / {total_kcs}")
print(f"KCs with shallow horizontal sequences (< 15 attempts/student): {len(short_sequence_kcs)} / {total_kcs}\n")

#
kc_summary = pd.DataFrame({
    'Success_Rate': kc_success,
    'Unique_Students': students_per_kc,
    'Avg_Sequence_Length': avg_sequence_per_kc
})


#print(kc_summary.head(10).to_string())
kc_summary.sort_values('Avg_Sequence_Length', ascending=False).head(15)

Validating Data to Original pyBKT Recommendations:
KCs with low vertical depth (< 100 students): 236 / 236
KCs with shallow horizontal sequences (< 15 attempts/student): 235 / 236



,Success_Rate,Unique_Students,Avg_Sequence_Length
primary_kc_id,,,
KC.U4.16.expected_value_calculation,0.557920,25,16.92
KC.U3.11.generalization_scope,0.568768,25,13.96
KC.U4.11.independent_events_probability,0.561728,25,12.96
KC.U2.03.conditional_relative_frequency,0.577778,25,9.00
KC.U4.01.probability_long_run_interpretation,0.633929,25,8.96
KC.U4.24.binomial_exact_probability,0.535874,25,8.92
KC.U9.08.regression_linearity_condition,0.592593,25,8.64
KC.U4.30.geometric_mean_sd,0.572500,25,8.00
KC.U1.22.normal_tail_interval_probability,0.575000,25,8.00


In [32]:
kc_summary.sort_values('Avg_Sequence_Length', ascending=False).head(15)

,Success_Rate,Unique_Students,Avg_Sequence_Length
primary_kc_id,,,
KC.U4.16.expected_value_calculation,0.557920,25,16.92
KC.U3.11.generalization_scope,0.568768,25,13.96
KC.U4.11.independent_events_probability,0.561728,25,12.96
KC.U2.03.conditional_relative_frequency,0.577778,25,9.00
KC.U4.01.probability_long_run_interpretation,0.633929,25,8.96
KC.U4.24.binomial_exact_probability,0.535874,25,8.92
KC.U9.08.regression_linearity_condition,0.592593,25,8.64
KC.U4.30.geometric_mean_sd,0.572500,25,8.00
KC.U1.22.normal_tail_interval_probability,0.575000,25,8.00


In [34]:
attempts_per_student_per_kc.sort_values('sequence_length', ascending=False).head(10)

,primary_kc_id,student_id,sequence_length
2541,KC.U4.16.expected_value_calculation,S011,17
2543,KC.U4.16.expected_value_calculation,S013,17
2550,KC.U4.16.expected_value_calculation,S020,17
2551,KC.U4.16.expected_value_calculation,S021,17
2552,KC.U4.16.expected_value_calculation,S022,17
2553,KC.U4.16.expected_value_calculation,S023,17
2554,KC.U4.16.expected_value_calculation,S024,17
2549,KC.U4.16.expected_value_calculation,S019,17
2555,KC.U4.16.expected_value_calculation,S025,17
2547,KC.U4.16.expected_value_calculation,S017,17


In [28]:
students_per_kc

primary_kc_id
KC.U1.02.observational_unit_variable                           25
KC.U1.03.variable_type_cat_quant                               25
KC.U1.04.variable_type_discrete_continuous                     25
KC.U1.05.categorical_freq_relative                             25
KC.U1.06.categorical_bar_pie_segmented                         25
                                                               ..
KC.U9.16.slope_test_pvalue_interpret                           25
KC.U9.17.slope_test_decision_conclusion                        25
KC.U9.18.scatterplot_for_regression_inference                  25
KC.U9.19.lsrl_equation_from_data                               25
KC.U9.20.regression_inference_scope_and_prediction_language    24
Name: student_id, Length: 236, dtype: int64

In [29]:
avg_sequence_per_kc

primary_kc_id
KC.U1.02.observational_unit_variable                           2.00
KC.U1.03.variable_type_cat_quant                               3.00
KC.U1.04.variable_type_discrete_continuous                     3.96
KC.U1.05.categorical_freq_relative                             6.96
KC.U1.06.categorical_bar_pie_segmented                         7.00
                                                               ... 
KC.U9.16.slope_test_pvalue_interpret                           2.88
KC.U9.17.slope_test_decision_conclusion                        3.84
KC.U9.18.scatterplot_for_regression_inference                  2.88
KC.U9.19.lsrl_equation_from_data                               2.88
KC.U9.20.regression_inference_scope_and_prediction_language    2.00
Name: sequence_length, Length: 236, dtype: float64